# Lab01 - Neural Networks

##  For the first part of this lab we will be comparing CNN and MLP on the mnist dataset

In [ ]:
import numpy
import torch
from torch import nn

import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
t_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)

val_dataset, test_dataset = torch.utils.data.random_split(t_dataset, [int(len(t_dataset)/2), int(len(t_dataset)/2)])

In [ ]:
BATCH_SIZE = 1024
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
img = next(iter(train_loader))
plt.imshow(img[0][0][0].numpy(), cmap='gray')

### Question 1: Implement the MLP class

Create a Multi-Layer Perceptron (MLP) with:
- An input layer that takes flattened 28x28 images (784 features)
- One hidden layer with ReLU activation
- An output layer for 10 classes

Hint: Use `nn.Linear` for the layers and `nn.ReLU` for activation.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        # TODO: Define the layers
        # self.layer1 = ...
        # self.relu = ...
        # self.layer2 = ...
        pass

    def forward(self, x):
        # TODO: Implement the forward pass
        pass

In [ ]:
mlp = MLP(input_size=28*28, hidden_size=128, output_size=10)
mlp.to(device)

In [ ]:
%%time
def train(model, train_loader, test_loader, num_epochs=10):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        for i, (images, labels) in enumerate(train_loader):
            images = images.view(images.shape[0], -1).to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

train(mlp, train_loader, test_loader, num_epochs=1)

In [ ]:
def test_model(model, val_dataloader, criterion, device):
    model.eval() 
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs = inputs.view(inputs.shape[0], -1).to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    print(f'Validation Loss: {avg_loss:.4f}, Validation Accuracy: {accuracy:.4f}')
    return avg_loss, accuracy

test_model(mlp, val_loader, criterion=torch.nn.CrossEntropyLoss(), device=device)

### Question 2: Implement the CNN class

Create a Convolutional Neural Network (CNN) with:
- A first convolutional layer: in_channels -> 32 filters, kernel_size=3, stride=1, padding=1
- A max pooling layer: kernel_size=2, stride=2
- A second convolutional layer: 32 -> 64 filters, kernel_size=3, stride=1, padding=1
- Another max pooling layer
- A fully connected layer: 64 * 7 * 7 -> 128
- An output layer: 128 -> num_classes
- Use ReLU activation between layers

Hint: Use `nn.Conv2d`, `nn.MaxPool2d`, and `nn.Linear`.

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()
        # TODO: Define the convolutional layers
        # self.conv1 = ...
        # self.pool = ...
        # self.conv2 = ...
        # self.fc1 = ...
        # self.fc2 = ...
        # self.relu = ...
        pass

    def forward(self, x):
        # TODO: Implement the forward pass
        # Remember to flatten before the fully connected layers
        pass
        
cnn_model = CNN(in_channels=1, num_classes=10).to(device)
print(cnn_model)

In [ ]:
def train_cnn(model, train_loader, test_loader, num_epochs=10):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device) 
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

train_cnn(cnn_model, train_loader, test_loader, num_epochs=1)

In [ ]:
def test_cnn(model, val_dataloader, criterion, device):
    model.eval() 
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs = inputs.to(device) # No flattening
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct_predictions / total_samples
    print(f'Validation Loss: {avg_loss:.4f}, Validation Accuracy: {accuracy:.4f}')
    return avg_loss, accuracy

test_cnn(cnn_model, val_loader, criterion=torch.nn.CrossEntropyLoss(), device=device)

## Part 2 — Residual Networks (ResNet)

So far, our CNN is *purely sequential*: every layer must learn a full transformation.

A **ResNet** adds a *skip connection* (also called a **residual connection**) so the network can learn a **residual**:

$$
y = F(x) + x
$$

This often makes optimization easier (especially for deeper networks) because gradients can flow through the identity path.

In this section you will:

1. Implement a small **ResNet** from scratch.
2. Upgrade the earlier `CNN` into a deeper `SimpleCNN`.
3. Extend `SimpleCNN` with **skip connections** (making it "ResNet-like").
4. Run a **small-scale convergence test** (quick training sanity check).
5. (Optional) Load a **pretrained ResNet** and fine-tune it on a small dataset.

### Question 3: Implement the ResidualBlock class

Create a Residual Block with:
- Two convolutional layers (3x3 kernel, padding=1)
- Batch normalization after each convolution
- ReLU activation
- A shortcut connection that adds the input to the output
- If the input and output channels differ (or stride != 1), use a 1x1 convolution for the shortcut

The forward pass should compute: `out = F(x) + shortcut(x)` followed by ReLU.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        # TODO: Define conv1, bn1, relu, conv2, bn2
        # self.conv1 = nn.Conv2d(...)
        # self.bn1 = nn.BatchNorm2d(...)
        # self.relu = nn.ReLU(inplace=True)
        # self.conv2 = nn.Conv2d(...)
        # self.bn2 = nn.BatchNorm2d(...)
        
        # TODO: Define the shortcut connection
        # If stride != 1 or in_channels != out_channels, use a 1x1 conv
        # self.shortcut = ...
        pass

    def forward(self, x):
        # TODO: Implement the forward pass with skip connection
        # out = conv1 -> bn1 -> relu -> conv2 -> bn2
        # out = out + shortcut(x)
        # out = relu(out)
        pass

### Question 4: Implement the ResNet18 class

Create a ResNet-18 style architecture with:
- Initial conv layer: in_channels -> 64, kernel_size=3, stride=1, padding=1
- Batch normalization and ReLU
- 4 layer groups using the ResidualBlock:
  - layer1: 64 channels, 2 blocks, stride=1
  - layer2: 128 channels, 2 blocks, stride=2
  - layer3: 256 channels, 2 blocks, stride=2
  - layer4: 512 channels, 2 blocks, stride=2
- Adaptive average pooling to (1, 1)
- Fully connected layer: 512 -> num_classes

Hint: Use the `_make_layer` helper method to create each layer group.

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super(ResNet18, self).__init__()
        self.in_channels = 64

        # TODO: Define the initial convolution, batch norm, and relu
        # self.conv1 = ...
        # self.bn1 = ...
        # self.relu = ...

        # TODO: Create the 4 layer groups using _make_layer
        # self.layer1 = self._make_layer(ResidualBlock, 64, 2, stride=1)
        # self.layer2 = ...
        # self.layer3 = ...
        # self.layer4 = ...

        # TODO: Define avgpool and fc layers
        # self.avgpool = ...
        # self.fc = ...
        pass

    def _make_layer(self, block, out_channels, num_blocks, stride):
        # TODO: Implement this helper method
        # First block may have stride > 1, rest have stride = 1
        # Update self.in_channels after each block
        pass

    def forward(self, x):
        # TODO: Implement the forward pass
        pass

resnet_small = ResNet18(in_channels=1, num_classes=10).to(device)
print(resnet_small)

### Upgrade: `SimpleCNN` with skip connections

Below we implement a deeper CNN (`SimpleCNN`) and then add skip connections to create a more ResNet-like network.

The key idea: if two tensors have the same shape, we can do `out = out + x` to create a residual path.

### Question 5: Implement the CNNWithSkips class

Create a CNN with skip connections:

**Block 1 (in_channels -> 32):**
- conv1a: in_channels -> 32, kernel_size=3, padding=1
- conv1b: 32 -> 32, kernel_size=3, padding=1
- skip1: If in_channels != 32, use a 1x1 conv to project, else use Identity

**Block 2 (32 -> 64):**
- conv2a: 32 -> 64, kernel_size=3, padding=1
- conv2b: 64 -> 64, kernel_size=3, padding=1
- skip2: 1x1 conv to project 32 -> 64

**Classifier:**
- MaxPool2d(2, 2) after each block
- fc1: 64 * 7 * 7 -> 128
- fc2: 128 -> num_classes

In forward: `out = relu(conv_output + skip(input))` for each block.

In [ ]:
class CNNWithSkips(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()

        # TODO: Define Block 1 (in_channels -> 32)
        # self.conv1a = ...
        # self.conv1b = ...
        # self.skip1 = ... (Identity or 1x1 conv)

        # TODO: Define Block 2 (32 -> 64)
        # self.conv2a = ...
        # self.conv2b = ...
        # self.skip2 = ...

        # TODO: Define pooling and classifier layers
        # self.pool = ...
        # self.fc1 = ...
        # self.fc2 = ...
        # self.relu = ...
        pass

    def forward(self, x):
        # TODO: Implement forward pass with skip connections
        # Block 1: residual = skip1(x), out = conv1a -> relu -> conv1b, out = relu(out + residual), pool
        # Block 2: residual = skip2(out), out = conv2a -> relu -> conv2b, out = relu(out + residual), pool
        # Classifier: flatten -> fc1 -> relu -> fc2
        pass


simple_cnn = CNN(in_channels=1, num_classes=10).to(device)
skip_cnn = CNNWithSkips(in_channels=1, num_classes=10).to(device)
print(simple_cnn)
print(skip_cnn)

### Small-scale convergence test

Before training "for real", it's useful to confirm that training **converges** on a tiny subset.

This helps you catch issues like:
- wrong shapes
- missing `.to(device)`
- learning rate too high/low
- model outputs not matching the loss

We'll train for a few epochs on a **small subset** and print loss/accuracy trends.

In [ ]:
from torch.utils.data import Subset
import numpy as np

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total


small_train = Subset(train_dataset, list(range(2000)))
small_val   = Subset(val_dataset,   list(range(1000)))

small_train_loader = torch.utils.data.DataLoader(small_train, batch_size=128, shuffle=True)
small_val_loader   = torch.utils.data.DataLoader(small_val,   batch_size=256, shuffle=False)

def quick_convergence_demo(model, epochs=3, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, small_train_loader, optimizer, criterion, device)
        va_loss, va_acc = eval_model(model, small_val_loader, criterion, device)
        print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f}, acc {tr_acc:.4f} | val loss {va_loss:.4f}, acc {va_acc:.4f}")

print("\n== CNN convergence check ==")
quick_convergence_demo(CNN(in_channels=1, num_classes=10).to(device), epochs=10, lr=1e-3)

print("\n== CNNWithSkips convergence check ==")
quick_convergence_demo(CNNWithSkips(in_channels=1, num_classes=10).to(device), epochs=10, lr=1e-3)

print("== ResNet18 convergence check ==")
quick_convergence_demo(ResNet18(in_channels=1, num_classes=10).to(device), epochs=10, lr=1e-3)

## Part 3 (Optional) — Transfer Learning with a Pretrained ResNet

If you have internet access in your environment (or the weights are already cached), you can load a **pretrained ResNet-18** from `torchvision` and fine-tune it on a small dataset.

We'll demonstrate using **CIFAR-10** (10 classes, 3-channel images). Since ResNet-18 was trained on ImageNet (224×224), we resize CIFAR images to 224×224 and apply ImageNet normalization.

**Goal:** Show that pretrained features transfer well and converge fast on small data.

In [ ]:
# NOTE:
# - This section uses torchvision.models pretrained weights.
# - If it fails due to download restrictions, you can skip it, or run it later on your own machine.

import torchvision
import torchvision.transforms as transforms

# ImageNet normalization constants
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

cifar_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=cifar_transform)
cifar_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=cifar_transform)

# small subset (for a quick lab run)
cifar_train_small = Subset(cifar_train, list(range(5000)))
cifar_test_small  = Subset(cifar_test,  list(range(1000)))

cifar_train_loader = torch.utils.data.DataLoader(cifar_train_small, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
cifar_test_loader  = torch.utils.data.DataLoader(cifar_test_small,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Load pretrained ResNet-18
weights = ResNet18_Weights.DEFAULT
pretrained = resnet18(weights=weights)

# Replace classification head for CIFAR-10
pretrained.fc = nn.Linear(pretrained.fc.in_features, 10)
pretrained = pretrained.to(device)

# Option A: freeze backbone (fast, strong baseline)
for name, p in pretrained.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, pretrained.parameters()), lr=1e-3)

for epoch in range(1, 3):  # 2 epochs for demo
    tr_loss, tr_acc = train_one_epoch(pretrained, cifar_train_loader, optimizer, criterion, device)
    te_loss, te_acc = eval_model(pretrained, cifar_test_loader, criterion, device)
    print(f"[Pretrained ResNet18] Epoch {epoch} | train acc {tr_acc:.4f} | test acc {te_acc:.4f}")

### What to submit

Add a short write-up answering:

1. **ResNet implementation:** Where did you add skip connections, and why does it help training?
2. **SimpleCNN → skip connections:** What changed structurally? Did it affect convergence speed on the small-scale test?
3. **Convergence evidence:** Paste the output logs (or a screenshot) showing loss decreasing and accuracy increasing.
4. **Pretrained ResNet demo (optional):** Report accuracy on the small CIFAR-10 subset and describe whether freezing vs fine-tuning helped.